# Tour: Reinforcing Recursive Language Models

Companion notebook to `tour.md` / `tour.tex`. Walks the reader from foundational background through the paper's concepts and into the proposed improvements. Every measurement-style improvement is reproduced inline.

Source: https://www.alphaxiv.org/blog/reinforcement-learning-for-rlms


## 1. Reader's contract

**Audience.** You are comfortable with policy gradients (Sutton 1999) and have read at least one PPO or GRPO paper. If not, drill into the chain repo first: `pleyva2004/first-principles-to-llms` Chapters 28-31.

**Time budget.** ~90 minutes for a careful pass; 30 minutes for a skim.

**Math-foundations entry-point matching your level:**
- Beginner → `pleyva2004/math-foundations/tours/cs-undergrad.md`
- Intermediate → start at chain Chapter 28 (SFT/RLHF/DPO)
- Advanced → jump straight to chain Chapter 31 (policy gradient + GRPO derivation)


## 2. Foundations walk (chain repo)

Topologically-ordered prereqs from `pleyva2004/first-principles-to-llms`:

| # | Chapter | Why this paper needs it | Pacing |
|---|---------|------------------------|--------|
| 25 | Causal masking; NTP loss as MLE | Every RLM rollout is a Ch 25 MDP at the token level | skim |
| 27 | Pre-training pipeline (tiny GPT) | Backbone here is Qwen3.5-4B pre-trained the same way; cold-start SFT reuses NTP loss | read |
| 28 | SFT, RLHF (PPO/GRPO), DPO | Cold-start SFT + GRPO are direct lifts | read |
| 29 | MDP foundations + Bellman | Trajectory + return + advantage definitions extend to the tree case | read |
| 30 | Max-entropy RL + soft Bellman | KL regularization to a reference policy = same closed-form trick | skim |
| 31 | Policy gradient + GRPO + RLHF/DPO bridge | The recursive-subtree loss is the chain Ch 31 GRPO objective lifted from sequence to tree | drill |


## 3. Paper concepts walk

Read in this order (see `learning-map/paper/concepts/<NN>-<slug>.md` for each):

1. RLM definition · 2. Python REPL agent environment · 3. Root rollout · 4. Child rollout (`rlm_query`)
5. RLM trajectory tree · 6. Shared policy (one model, both roles) · 7. PPO clipped surrogate
8. Group-relative advantage (GRPO) · 9. Advantage inheritance child-from-parent · 10. $1/k_g$ child contribution averaging
11. Recursive subtree loss (arbitrary depth) · 12. Evidence selection task · 13. Rubric-based LLM-judge reward · 14. Cold-start SFT for RLM harness syntax


## 4. Improvements walk

Each proposal is validated by either a **PROOF** (`proofs/<slug>.tex`) or a **MEASUREMENT** (`improvements/<slug>.py`). The measurement scripts are run inline below.


### 4.1 Tighter unbiasedness theorem for advantage inheritance

**Validation: PROOF** — `proofs/inheritance-unbiasedness.tex`

Formalises the conditional-independence assumption that makes the blog's informal unbiasedness claim rigorous. Compile the proof PDF for the full theorem + discussion of what breaks the assumption.


### 4.2 Per-child local-baseline variance reduction

**Validation: MEASUREMENT** — `improvements/local-baseline.py`

Adds a learned local critic $b_\phi(s_{g,i})$ for each child, so the per-child loss uses the corrected advantage $A_g + (\hat V^{\text{loc}}_{g,i} - b_\phi(s_{g,i}))$. Should reduce per-step gradient variance.


In [ ]:
import sys, importlib.util, json
spec = importlib.util.spec_from_file_location('local_baseline', '/home/pleyv/ai-research-studies/reinforcing-recursive-language-models/improvements/local-baseline.py')
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
results = mod.measure()
print(json.dumps(results, indent=2))
print()
print('Variance ratio (lower = baseline helps):', results.get('ratio'))


### 4.3 $k_g$ scale-law experiment

**Validation: MEASUREMENT** — `improvements/k-scale-sweep.py`

Holds total compute fixed: $G \cdot (1 + k_g) = $ const. Sweeps $k_g \in \{1, 2, 4, 8\}$. Reports final reward per $k_g$ to locate the knee.


In [ ]:
import importlib.util, json
spec = importlib.util.spec_from_file_location('k_scale_sweep', '/home/pleyv/ai-research-studies/reinforcing-recursive-language-models/improvements/k-scale-sweep.py')
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
results = mod.measure()
print(json.dumps(results, indent=2))


### 4.4 RLM children as options (Sutton-Precup-Singh correspondence)

**Validation: PROOF** — `proofs/rlm-as-options.tex`

Maps each `rlm_query(prompt)` invocation to an option $o = (I, \pi_o, \beta)$ in the SMDP framework. Identifies which Sutton-Precup-Singh theorems carry over (intra-option Q-learning convergence, hierarchical Bellman equations).


## 5. What to do next

Three concrete action items, in order of leverage:

1. **Run the toy sandbox** (`sandbox/toy_recursive_bandit.py`) and verify the shared-policy training matches the paper's qualitative claim (parent + children both learn).
2. **Pick the most personally promising improvement.** For me, that's the $k_g$ scale law (4.3) — concrete, cheap to test, and directly informs hyperparameter choice for any follow-up RLM training.
3. **Sketch the depth-2 generalization** of the recursive-subtree loss (root → children → grandchildren). The math is in section 2.4 of `02-math-deep-dive.md`; what's missing is an empirical demonstration that the inherited-advantage variance doesn't explode.

If you want to take this further: the options-framework connection (4.4) opens up hierarchical-RL theorems that could give principled answers to several of the open questions in `02-math-deep-dive.md` Section 7.
